# Particle Filter — Bearings-Only Tracking

## Why do we need a Particle Filter?

The Kalman Filter and Extended Kalman Filter both assume that uncertainty can be represented as a **Gaussian distribution**. This works well when:
- The process model is linear (KF), or mildly nonlinear near the current estimate (EKF)
- The posterior state distribution is roughly bell-shaped

But consider this scenario: a sensor can only measure the **bearing** (angle) to a target — not the distance. After the first measurement, all we know is that the target lies *somewhere along a ray*. That is a fundamentally **non-Gaussian** distribution — it is elongated along the ray and has no meaningful single peak in range.

The **Particle Filter (PF)** makes no Gaussian assumption. It represents the posterior as a cloud of weighted samples (particles), and can capture any shape of distribution — multimodal, banana-shaped, ring-shaped, or otherwise.

---

## The Problem: Bearings-Only Tracking

- A **fixed sensor** sits at the origin and measures only the **bearing angle** $\theta$ to a moving target
- The **target** moves with approximately constant velocity
- We do not know the initial range to the target

This is a classic problem in sonar tracking (a submarine that can hear bearing but not range), radar tracking, and passive optical sensors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

## Process Model

The target moves with a constant velocity, subject to small random accelerations:

$$
\mathbf{x}_{k+1} = F \mathbf{x}_k + \mathbf{w}_k, \quad \mathbf{w}_k \sim \mathcal{N}(0, Q)
$$

where the state is $\mathbf{x} = [p_x, v_x, p_y, v_y]^\top$ and

$$
F = \begin{bmatrix} 1 & \Delta t & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & \Delta t \\ 0 & 0 & 0 & 1 \end{bmatrix}
$$

The process noise comes from unknown accelerations $a \sim \mathcal{N}(0, \sigma_a^2)$, giving:

$$
Q = \sigma_a^2 \begin{bmatrix}
\frac{\Delta t^4}{4} & \frac{\Delta t^3}{2} & 0 & 0 \\
\frac{\Delta t^3}{2} & \Delta t^2 & 0 & 0 \\
0 & 0 & \frac{\Delta t^4}{4} & \frac{\Delta t^3}{2} \\
0 & 0 & \frac{\Delta t^3}{2} & \Delta t^2
\end{bmatrix}
$$

## Measurement Model

The sensor measures only the **bearing angle** $\theta$ from the sensor position $\mathbf{s} = (s_x, s_y)$ to the target:

$$
z_k = h(\mathbf{x}_k) + \nu_k = \arctan\!\left(\frac{p_y - s_y}{p_x - s_x}\right) + \nu_k, \quad \nu_k \sim \mathcal{N}(0, \sigma_\theta^2)
$$

This measurement function is **highly nonlinear**. Crucially, a single bearing measurement gives no information about **range** — the target could be near or far along the ray. This means the posterior after one measurement is a long, thin strip along a ray — decidedly non-Gaussian.

### Why EKF struggles here

The EKF linearises $h$ around the current state estimate and forces the posterior to be Gaussian. When the true posterior is arc-shaped or multimodal, this approximation is poor and the filter can diverge. The Particle Filter makes no such assumption.

In [ ]:
# ============================================================
#  Simulation
# ============================================================

def simulate_truth(N, dt, x0):
    """Constant-velocity truth trajectory. State: [px, vx, py, vy]"""
    states = np.zeros((N, 4))
    states[0] = x0
    F = np.array([
        [1, dt, 0,  0 ],
        [0,  1, 0,  0 ],
        [0,  0, 1, dt ],
        [0,  0, 0,  1 ]
    ])
    for k in range(1, N):
        states[k] = F @ states[k - 1]
    return states


def simulate_bearings(true_states, sensor_pos, sigma_theta):
    """Generate noisy bearing measurements for each timestep."""
    N = len(true_states)
    z = np.zeros(N)
    for k in range(N):
        dx = true_states[k, 0] - sensor_pos[0]
        dy = true_states[k, 2] - sensor_pos[1]
        z[k] = np.arctan2(dy, dx) + np.random.randn() * sigma_theta
    return z

## Particle Filter Implementation

We use the **Sequential Importance Resampling (SIR)** filter, also called the Bootstrap Particle Filter. Each iteration has three steps:

**1. Predict** — propagate each particle through the dynamics:
$$
\mathbf{x}_k^{(i)} \sim p(\mathbf{x}_k \mid \mathbf{x}_{k-1}^{(i)})
$$
In practice: apply $F$ to each particle and add sampled process noise.

**2. Update** — weight each particle by how well it explains the measurement:
$$
w_k^{(i)} \propto w_{k-1}^{(i)} \cdot p(z_k \mid \mathbf{x}_k^{(i)})
$$
For Gaussian bearing noise: $p(z \mid \mathbf{x}) = \mathcal{N}(z;\, h(\mathbf{x}),\, \sigma_\theta^2)$.

**3. Resample** — when the effective sample size $N_{\text{eff}} = \left(\sum_i (w^{(i)})^2\right)^{-1}$ drops too low, draw $N$ new particles with replacement according to the current weights. This prevents weight degeneracy. We use **systematic resampling** for lower variance.

**Estimate** — the posterior mean is the weighted average of particles:
$$
\hat{\mathbf{x}}_k = \sum_{i=1}^N w_k^{(i)} \mathbf{x}_k^{(i)}
$$

In [ ]:
# ============================================================
#  Particle Filter
# ============================================================

def pf_init(N_particles, px_init, py_init, vx_std, vy_std):
    """
    Initialise particles spread along the first bearing ray.
    We do NOT know the range, so we spread particles uniformly in range
    and give each a random velocity drawn from a broad prior.
    """
    particles = np.column_stack([
        px_init,
        np.random.randn(N_particles) * vx_std,
        py_init,
        np.random.randn(N_particles) * vy_std,
    ])
    weights = np.ones(N_particles) / N_particles
    return particles, weights


def pf_predict(particles, dt, sigma_a):
    """
    Propagate each particle with constant-velocity dynamics
    plus random acceleration noise.
    """
    N = len(particles)
    ax = np.random.randn(N) * sigma_a
    ay = np.random.randn(N) * sigma_a

    new_p = particles.copy()
    new_p[:, 0] = particles[:, 0] + particles[:, 1] * dt + 0.5 * ax * dt ** 2
    new_p[:, 1] = particles[:, 1] + ax * dt
    new_p[:, 2] = particles[:, 2] + particles[:, 3] * dt + 0.5 * ay * dt ** 2
    new_p[:, 3] = particles[:, 3] + ay * dt
    return new_p


def pf_update(particles, weights, z, sensor_pos, sigma_theta):
    """
    Reweight particles by Gaussian bearing likelihood.
    Handles angle wrapping to keep innovations in [-pi, pi].
    """
    dx = particles[:, 0] - sensor_pos[0]
    dy = particles[:, 2] - sensor_pos[1]
    predicted_bearing = np.arctan2(dy, dx)

    innovation = z - predicted_bearing
    innovation = (innovation + np.pi) % (2 * np.pi) - np.pi  # wrap to [-pi, pi]

    likelihood = np.exp(-0.5 * (innovation / sigma_theta) ** 2)
    weights = weights * likelihood

    total = weights.sum()
    if total < 1e-300:           # all particles collapsed — reset
        weights = np.ones(len(particles)) / len(particles)
    else:
        weights /= total

    return weights


def effective_sample_size(weights):
    """N_eff = 1 / sum(w^2).  Full diversity = N; full degeneracy = 1."""
    return 1.0 / np.sum(weights ** 2)


def systematic_resample(weights):
    """Systematic resampling — O(N) and lower variance than multinomial."""
    N = len(weights)
    positions = (np.arange(N) + np.random.uniform(0, 1)) / N
    cumsum = np.cumsum(weights)
    return np.searchsorted(cumsum, positions)


def pf_resample(particles, weights, resample_threshold=0.5):
    """Resample when N_eff < threshold * N to combat weight degeneracy."""
    N = len(weights)
    if effective_sample_size(weights) < resample_threshold * N:
        indices = systematic_resample(weights)
        particles = particles[indices]
        weights = np.ones(N) / N
    return particles, weights


def pf_estimate(particles, weights):
    """Weighted mean of the particle cloud."""
    return np.sum(particles * weights[:, np.newaxis], axis=0)

## EKF for Comparison

For the EKF we linearise the bearing measurement model:

$$
H = \frac{\partial h}{\partial \mathbf{x}} = \left[
\frac{-(p_y - s_y)}{r^2},\; 0,\; \frac{p_x - s_x}{r^2},\; 0
\right]
$$

where $r^2 = (p_x - s_x)^2 + (p_y - s_y)^2$. The EKF then proceeds with the standard predict/update equations, but using $H$ in place of the true nonlinear measurement model.

In [ ]:
# ============================================================
#  EKF for Bearings-Only Tracking
# ============================================================

def bearing_h(x, sensor_pos):
    """Nonlinear bearing measurement function."""
    dx = x[0] - sensor_pos[0]
    dy = x[2] - sensor_pos[1]
    return np.arctan2(dy, dx)


def bearing_H_jacobian(x, sensor_pos):
    """Jacobian of bearing w.r.t. state [px, vx, py, vy]."""
    dx = x[0] - sensor_pos[0]
    dy = x[2] - sensor_pos[1]
    r2 = dx ** 2 + dy ** 2
    return np.array([[-dy / r2, 0.0, dx / r2, 0.0]])


def ekf_predict(x, P, F, Q):
    x_pred = F @ x
    P_pred = F @ P @ F.T + Q
    return x_pred, P_pred


def ekf_update(x_pred, P_pred, z, sensor_pos, R):
    H = bearing_H_jacobian(x_pred, sensor_pos)
    z_pred = bearing_h(x_pred, sensor_pos)

    # Innovation with angle wrapping
    innov = z - z_pred
    innov = (innov + np.pi) % (2 * np.pi) - np.pi

    S = H @ P_pred @ H.T + R
    K = P_pred @ H.T / S[0, 0]

    x_upd = x_pred + K.flatten() * innov
    I = np.eye(4)
    P_upd = (I - np.outer(K.flatten(), H.flatten())) @ P_pred
    P_upd = 0.5 * (P_upd + P_upd.T)   # enforce symmetry

    return x_upd, P_upd

In [ ]:
# ============================================================
#  Simulation Setup
# ============================================================

np.random.seed(0)

dt         = 1.0             # seconds per step
N_steps    = 150             # total timesteps
N_particles = 3000           # number of particles

sensor_pos  = np.array([0.0, 0.0])
sigma_theta = np.deg2rad(1.5)  # 1.5 degree bearing noise
sigma_a     = 0.3              # acceleration noise (m/s^2)

# True initial state [px, vx, py, vy]
# Target starts ~1000m away, moving at 5 m/s east, 3 m/s north
x0_true = np.array([800.0, 5.0, 600.0, 3.0])

# Simulate ground truth and measurements
true_states  = simulate_truth(N_steps, dt, x0_true)
measurements = simulate_bearings(true_states, sensor_pos, sigma_theta)

print(f"True initial range: {np.hypot(x0_true[0], x0_true[2]):.0f} m")
print(f"True initial bearing: {np.rad2deg(np.arctan2(x0_true[2], x0_true[0])):.1f} deg")
print(f"Bearing noise std: {np.rad2deg(sigma_theta):.1f} deg")

In [ ]:
# ============================================================
#  Particle Filter Initialisation
#
#  Key insight: we only have one bearing measurement initially.
#  We spread particles uniformly in range (200 – 2500 m) along
#  that ray. This is the correct non-Gaussian prior.
# ============================================================

first_bearing = measurements[0]

# Uniform in range, small noise perpendicular to ray
init_ranges = np.random.uniform(200, 2500, N_particles)
px_init = init_ranges * np.cos(first_bearing) + np.random.randn(N_particles) * 30
py_init = init_ranges * np.sin(first_bearing) + np.random.randn(N_particles) * 30

particles, weights = pf_init(N_particles, px_init, py_init, vx_std=5.0, vy_std=5.0)

# ============================================================
#  EKF Initialisation
#
#  EKF needs a Gaussian initial state. We centre it on the
#  first bearing at an assumed range of 1000 m — but with
#  large variance to reflect our range uncertainty.
# ============================================================

assumed_range = 1000.0
x_ekf = np.array([
    assumed_range * np.cos(first_bearing),
    0.0,
    assumed_range * np.sin(first_bearing),
    0.0
])
# High range uncertainty but zero knowledge of velocity
P_ekf = np.diag([800**2, 8**2, 800**2, 8**2])

# Matrices for EKF
F = np.array([
    [1, dt, 0,  0],
    [0,  1, 0,  0],
    [0,  0, 1, dt],
    [0,  0, 0,  1]
])

Q_ekf = sigma_a**2 * np.array([
    [dt**4/4, dt**3/2,      0,       0],
    [dt**3/2,    dt**2,      0,       0],
    [      0,       0, dt**4/4, dt**3/2],
    [      0,       0, dt**3/2,    dt**2]
])

R_ekf = np.array([[sigma_theta**2]])

In [ ]:
# ============================================================
#  Run Both Filters
# ============================================================

pf_estimates  = []
ekf_estimates = []
n_eff_log     = []

# Save particle snapshots for visualisation
snapshot_steps = [0, 5, 20, 50, 100, 149]
snapshots      = {}   # step -> (particles, weights)

for k in range(N_steps):
    # --- Particle Filter ---
    particles = pf_predict(particles, dt, sigma_a)
    weights   = pf_update(particles, weights, measurements[k], sensor_pos, sigma_theta)
    n_eff_log.append(effective_sample_size(weights))

    if k in snapshot_steps:
        snapshots[k] = (particles.copy(), weights.copy())

    particles, weights = pf_resample(particles, weights)
    pf_estimates.append(pf_estimate(particles, weights))

    # --- EKF ---
    x_ekf, P_ekf = ekf_predict(x_ekf, P_ekf, F, Q_ekf)
    x_ekf, P_ekf = ekf_update(x_ekf, P_ekf, measurements[k], sensor_pos, R_ekf)
    ekf_estimates.append(x_ekf.copy())

pf_estimates  = np.array(pf_estimates)
ekf_estimates = np.array(ekf_estimates)

print("Filters complete.")
print(f"Mean N_eff over run: {np.mean(n_eff_log):.0f} / {N_particles}")

In [ ]:
# ============================================================
#  Plot 1: Trajectory Comparison
# ============================================================

# Convert bearing measurements to a plottable ray for each timestep
r_ray = 3500

fig, ax = plt.subplots(figsize=(10, 8))

# Draw a few bearing rays
for k in range(0, N_steps, 15):
    bx = r_ray * np.cos(measurements[k])
    by = r_ray * np.sin(measurements[k])
    ax.plot([0, bx], [0, by], color='gold', alpha=0.2, linewidth=0.8)

ax.plot(true_states[:, 0], true_states[:, 2],
        'g-', linewidth=2.5, label='True trajectory', zorder=4)
ax.plot(pf_estimates[:, 0], pf_estimates[:, 2],
        'b-', linewidth=2, label='PF estimate', zorder=3)
ax.plot(ekf_estimates[:, 0], ekf_estimates[:, 2],
        'r--', linewidth=2, label='EKF estimate', zorder=3)

# Mark start and end
ax.scatter(*true_states[0, [0, 2]], s=120, c='green', marker='o', zorder=5)
ax.scatter(*true_states[-1, [0, 2]], s=120, c='green', marker='s', zorder=5)
ax.scatter([0], [0], s=250, c='black', marker='^', zorder=5, label='Sensor')

ax.set_xlabel('X position (m)')
ax.set_ylabel('Y position (m)')
ax.set_title('Bearings-Only Tracking: Particle Filter vs EKF\n'
             '(gold lines = bearing measurements)')
ax.legend()
ax.set_aspect('equal')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  Plot 2: Particle Cloud Evolution
#  Shows the non-Gaussian posterior at different timesteps
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, step in enumerate(snapshot_steps):
    ax = axes[i]
    snap_particles, snap_weights = snapshots[step]

    # Bearing ray at this timestep
    bearing = measurements[step]
    ax.plot([0, r_ray * np.cos(bearing)], [0, r_ray * np.sin(bearing)],
            color='gold', alpha=0.6, linewidth=2, label='Bearing')

    # Particle cloud (size proportional to weight)
    max_w = snap_weights.max()
    sizes = np.clip(500 * snap_weights / (max_w + 1e-300), 1, 80)
    ax.scatter(snap_particles[:, 0], snap_particles[:, 2],
               s=sizes, alpha=0.25, c='steelblue', label='Particles')

    # True position
    ax.scatter(true_states[step, 0], true_states[step, 2],
               s=150, c='green', marker='*', zorder=5, label='True')

    # PF estimate
    ax.scatter(pf_estimates[step, 0], pf_estimates[step, 2],
               s=80, c='blue', marker='D', zorder=5, label='PF est')

    # EKF estimate
    ax.scatter(ekf_estimates[step, 0], ekf_estimates[step, 2],
               s=80, c='red', marker='x', linewidths=2, zorder=5, label='EKF est')

    # Sensor
    ax.scatter([0], [0], s=120, c='black', marker='^', zorder=5)

    ax.set_xlim(-300, 3500)
    ax.set_ylim(-300, 2500)
    ax.set_title(f'$t = {step}$ s', fontsize=12)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.grid(True)
    if i == 0:
        ax.legend(fontsize=8, loc='upper right')

plt.suptitle('Particle Cloud Evolution (non-Gaussian posterior along bearing ray)',
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  Plot 3: Position Error Over Time
# ============================================================

pf_error  = np.sqrt((pf_estimates[:, 0]  - true_states[:, 0])**2 +
                    (pf_estimates[:, 2]  - true_states[:, 2])**2)
ekf_error = np.sqrt((ekf_estimates[:, 0] - true_states[:, 0])**2 +
                    (ekf_estimates[:, 2] - true_states[:, 2])**2)

# Compute RMSE over the second half (post-convergence)
half = N_steps // 2
pf_rmse_late  = np.sqrt(np.mean(pf_error[half:]**2))
ekf_rmse_late = np.sqrt(np.mean(ekf_error[half:]**2))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(pf_error,  'b-',  linewidth=1.5, label=f'PF  (late RMSE = {pf_rmse_late:.1f} m)')
ax1.plot(ekf_error, 'r--', linewidth=1.5, label=f'EKF (late RMSE = {ekf_rmse_late:.1f} m)')
ax1.set_ylabel('Position error (m)')
ax1.set_title('Position Error and Effective Sample Size')
ax1.legend()
ax1.grid(True)

ax2.plot(n_eff_log, 'purple', linewidth=1.5)
ax2.axhline(0.5 * N_particles, color='grey', linestyle='--',
            label=f'Resample threshold ({0.5 * N_particles:.0f})')
ax2.set_xlabel('Time step')
ax2.set_ylabel('$N_{\\mathrm{eff}}$')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"PF  RMSE (full run):  {np.sqrt(np.mean(pf_error**2)):.1f} m")
print(f"EKF RMSE (full run):  {np.sqrt(np.mean(ekf_error**2)):.1f} m")
print(f"PF  RMSE (t > {half}):  {pf_rmse_late:.1f} m")
print(f"EKF RMSE (t > {half}):  {ekf_rmse_late:.1f} m")

In [ ]:
# ============================================================
#  Plot 4: Velocity Estimation
#  Bearings-only tracking also recovers velocity — shown here
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

labels = ['$p_x$ (m)', '$v_x$ (m/s)', '$p_y$ (m)', '$v_y$ (m/s)']
state_idx = [0, 1, 2, 3]

for ax, idx, label in zip(axes.flatten(), state_idx, labels):
    # True state index: px=0, vx=1, py=2, vy=3
    true_col = [0, 1, 2, 3][idx]  # same ordering
    ax.plot(true_states[:, true_col], 'g-', linewidth=2, label='True')
    ax.plot(pf_estimates[:, idx],  'b-',  linewidth=1.5, label='PF')
    ax.plot(ekf_estimates[:, idx], 'r--', linewidth=1.5, label='EKF')
    ax.set_ylabel(label)
    ax.set_xlabel('Time step')
    ax.legend(fontsize=8)
    ax.grid(True)

plt.suptitle('State Estimates: Particle Filter vs EKF', fontsize=13)
plt.tight_layout()
plt.show()

## Analysis

### What happened at $t=0$?

With only a single bearing measurement, neither filter knows the range. The particle filter represents this correctly: the particle cloud is spread along the bearing ray, reflecting the **true non-Gaussian posterior**. The EKF is forced to represent this as a Gaussian, which is an approximation.

### Convergence

As more bearing measurements arrive from slightly different directions (because the target moves), the **bearing geometry** becomes informative about range. The particle cloud progressively collapses around the true trajectory. This is called **observability** — the target becomes observable because its motion creates a changing bearing.

### EKF behaviour

The EKF linearises the bearing model at each step. When the initial uncertainty is large, this linearisation can be inaccurate, causing the filter to be slow to converge or inconsistent. The particle filter avoids this by propagating the full nonlinear model through the particle cloud.

### When to use a Particle Filter

| Scenario | KF | EKF | PF |
|---|---|---|---|
| Linear dynamics, Gaussian noise | ✅ Optimal | — | Works but overkill |
| Mildly nonlinear, Gaussian noise | ❌ | ✅ Good approx | Works |
| Highly nonlinear OR non-Gaussian posterior | ❌ | ⚠️ May diverge | ✅ |
| Multimodal distribution (e.g., target ambiguity) | ❌ | ❌ | ✅ |

Bearings-only tracking falls squarely in the last two rows — hence the particle filter is the right tool.